This notebook performs structured extraction of entities and relationships from hydrology research papers
using a predefined ontology and the OpenAI API. It involves three main stages:

1. Generate ontology-aligned entity and relationship definitions.
2. Extract metadata-based entities for each paper.
3. Extract text-based entities and intra-paper relationships using few-shot prompting.

Requirements:
- Ensure the following files are in place:
    - ontology.txt
    - extracted_results_with_context.json
    - metadata.json
    - entity_to_category_map.json
    - entity_relationship_map.json

- Set your OpenAI API key in a .env file as: OPENAI_API_KEY=your_key_here

In [ ]:
# Import libraries
import openai
import json
import re
import os
from dotenv import load_dotenv

In [ ]:
# Load API key from .env file
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
#OPENAI_API_KEY = os.getenv("ANDRES_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OpenAI API Key is missing in the .env file.")

# Initialize OpenAI client
client = openai.OpenAI(api_key=OPENAI_API_KEY)

In [ ]:
# Define prompts for extracting entities and relationships
prompt_role = "You are an expert in knowledge graph construction."

prompt_entities = """
Extract all entities (nodes) present in the ontology. Each entity should include:

- `name`: The entity's name.
- `description`: A brief definition of the entity.
- `macro_class`: The high-level category the entity belongs to.
- `category`: The macro class it belongs to.
- `possible_properties`: A list of **intrinsic attributes** (internal properties) that describe characteristics of the entity itself, rather than its relationships with other entities.

**Guidelines for Assigning `possible_properties`:**
- If the entity represents a **research component** (e.g., "Research Problem", "Main Hypothesis"), include attributes like:
  - `"description"`, `"importance_level"`, `"assumptions"`, `"limitations"`
- If the entity represents **data or variables** (e.g., "Key Hydrological Variables"), include attributes like:
  - `"unit_of_measurement"`, `"temporal_resolution"`, `"spatial_resolution"`, `"source"`
- If the entity is **methodology-related** (e.g., "Simulation Models", "Computational Tools"), include attributes like:
  - `"model_type"`, `"accuracy_metrics"`, `"software_dependency"`, `"runtime_complexity"`
- If the entity is **findings-related** (e.g., "Quantitative Findings", "Identified Patterns"), include attributes like:
  - `"statistical_significance"`, `"data_sources"`, `"visual_representation_type"`
- If the entity represents **theoretical or geographical context** (e.g., "Region of Study", "Hydrological Theories"), include attributes like:
  - `"geographical_scope"`, `"climate_zone"`, `"historical_relevance"`

**Important Constraints:**
- The properties should **not** be relationships between entities (e.g., `"solves"`, `"measured_in"`), which belong to a separate relationship extraction process.
- Ensure the output is a valid JSON object, following this structure:
```json
{
    "entities": [
        {
            "name": "EntityName",
            "description": "Brief description",
            "macro_class": "Macro Class Name",
            "category": "Category it belongs to",
            "possible_properties": ["Property1", "Property2", "Property3", ...]
        }
    ]
}

"""

prompt_relationships = """
Extract all relationships between entities from the given ontology and entity definitions. Each relationship should include:
- `relationship_name`: The name of the relationship.
- `source_entity`: The entity where the relationship starts.
- `target_entity`: The entity where the relationship ends.
- `description`: A brief definition of the relationship.
- `possible_properties`: A list of possible properties this relationship might have.

**Important Considerations:**
1. **Ensure that every entity appears in at least one relationship** (no entity should be isolated).  
2. **Cover all predefined relationships from the ontology.**  
3. **Expand logical relationships for hydrological concepts** (e.g., `"influences"`, `"depends on"` should connect variables like *Streamflow*, *Evapotranspiration*, *Soil Moisture*).  
4. **Connect "Academic Paper" to its components**:
   - `"has_component"` should connect *Academic Paper* to *Research Questions*, *Findings*, *Methodology*, etc.
   - `"cites"` should connect *Academic Paper* to *Academic Paper*.

### Guidelines for Relationships:
- If an entity represents a **research component**, connect it via `"investigates"`, `"builds on"`, `"extends"`, `"contradicts"`.
- If an entity represents **data or variables**, connect it via `"measured in"`, `"used in"`, `"validated by"`, `"collected from"`.
- If an entity represents **methodology**, connect it via `"applies to"`, `"analyzed using"`, `"tested using"`, `"calibrated using"`.
- If an entity is **findings-related**, connect it via `"derived from"`, `"represented in"`, `"supports"`, `"contradicts"`.
- If an entity is **hydrology-specific**, connect it via `"influences"`, `"modifies"`, `"depends on"`, `"affects"`.

### Ensure the output is a valid JSON object:
{
    "relationships": [
        {
            "relationship_name": "RELATIONSHIP_NAME",
            "source_entity": "EntityName",
            "target_entity": "EntityName",
            "description": "Brief description",
            "possible_properties": ["Property1", "Property2"]
        }
    ]
}
}
"""



In [ ]:
# Load ontology
try:
    with open("ontology.txt", "r", encoding="utf-8") as file:
        ontology = file.read().strip()
except FileNotFoundError:
    raise FileNotFoundError("The ontology.txt file is missing. Please create it and add the ontology text.")

# Construct chat messages for entities
mess_entities = [
    {"role": "system", "content": prompt_role},
    {"role": "user", "content": "Given the following ontology definition:"},
    {"role": "user", "content": ontology},
    {"role": "user", "content": prompt_entities},
]

In [ ]:
def extract_data(messages, type):

    results = {
        type: {}
    }

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0.4
        )

        extracted_data = json.loads(response.choices[0].message.content)

        # Store the extracted data in the results dictionary
        results[type] = extracted_data.get(type, {})

    except json.JSONDecodeError:
        print("Error parsing JSON response.")
    except Exception as e:
        print(f"Error extracting category data: {e}")

    return results


In [ ]:
# Extract entities
entities_data = extract_data(mess_entities, "entities")

# Save results to JSON files
if entities_data:
    with open("entities.json", "w", encoding="utf-8") as f:
        json.dump(entities_data, f, indent=4)

print("Entities extracted and saved.")

In [ ]:
# Load entities
try:
    with open("entities.json", "r", encoding="utf-8") as file:
        entities = file.read().strip()
except FileNotFoundError:
    raise FileNotFoundError("The entities.json file is missing. Please create it and add the entities in json format.")

# Construct chat messages for relationships
mess_relationships = [
    {"role": "system", "content": prompt_role},
    {"role": "user", "content": "Given the following ontology definition and entity list in json format:"},
    {"role": "user", "content": ontology},
    {"role": "user", "content": entities},
    {"role": "user", "content": prompt_relationships},
]

In [ ]:
# Extract relationships

relationships_data = extract_data(mess_relationships, "relationships")

if relationships_data:
    with open("relationships.json", "w", encoding="utf-8") as f:
        json.dump(relationships_data, f, indent=4)

print("Relationships extracted and saved.")

In [ ]:
# Paths
METADATA_PATH           = "metadata.json"
EXTRACTED_EXAMPLES_PATH = "extracted_results_examples.json"
EXTRACTED_PATH          = "extracted_results_with_context.json"
ENTITIES_DEF_PATH       = "entities.json"
ENTITY_MAP_PATH         = "entity_to_category_map.json"
EXAMPLE_ENTITIES_PATH   = "entity_examples.json"
EXTRACTED_TEST_PATH     = "extracted_test_results.json"
ENTITY_REL_MAP_PATH     = "entity_relationship_map.json"



### Generate examples of entities

In [ ]:
# Load data files
with open(ENTITIES_DEF_PATH , "r", encoding="utf-8") as f:
    entities_data = json.load(f)["entities"]

with open(ENTITY_MAP_PATH, "r", encoding="utf-8") as f:
#with open(SINGLE_CAT_MAP_PATH, "r", encoding="utf-8") as f:
    entity_to_category_map = json.load(f)

with open(EXTRACTED_EXAMPLES_PATH, "r", encoding="utf-8") as f:
    extracted_examples = json.load(f)

# Organize paper content by ID
paper_texts_by_id = {}
for paper in extracted_examples:
    pid = paper.get("Paper ID", "unknown")
    paper_texts_by_id[pid] = {k: v.strip() for k, v in paper.items() if k != "Paper ID"}

# Prompt generator for a single example
def generate_example_prompt(entity_name, properties, input_text):
    property_list = ', '.join(properties)
    properties_block = ',\n        '.join([f'"{prop}": "..."' for prop in properties])

    prompt = f"""You are an expert in hydrology and scientific knowledge extraction for knowledge graph construction.

Your task is to extract all relevant **{entity_name}** entities from the following input. Each entity should be concise and meaningful. Also extract any of these properties if available: {property_list}. If no entity is found, return an empty list.

What to do:
- Identify all relevant entities of type **{entity_name}** in each example input.
- Return each entity as a short, semantically meaningful name.
- Avoid copying long sentences or including locations/datasets unless essential for the entity **{entity_name}**.
- Also extract these properties if available: {property_list}
- If no relevant entities are found, respond with an empty list.
- If importance_level is part of the property list, and it is not present in the input, infer it from the context.

Return only the following JSON format:
{{
  "entities": [
    {{
      "name": "...",
      "category": "{entity_name}",
      "properties": {{
        {properties_block}
      }}
    }}
  ]
}}

### Now your turn

Input text:
\"\"\"{input_text}\"\"\"
"""
    return prompt

# Main processing
final_results = []

for entity in entities_data:
    name = entity["name"]
    properties = entity["possible_properties"]
    categories = entity_to_category_map.get(name, [])

    if "Metadata" in categories:
        continue  # Skip metadata-based entities

    paper_examples = []

    for paper_id, content_by_category in paper_texts_by_id.items():
        relevant_text = "\n".join([content_by_category[c] for c in categories if c in content_by_category])
        if not relevant_text:
            continue

        prompt = generate_example_prompt(name, properties, relevant_text)

        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.2,
                response_format={"type": "json_object"}
            )

            parsed_output = response.choices[0].message.content.strip()
            paper_examples.append({
                "input": relevant_text,
                "expected_output": json.loads(parsed_output)
            })

        except Exception as e:
            paper_examples.append({
                "input": relevant_text,
                "expected_output": f"ERROR: {str(e)}"
            })

    if paper_examples:
        final_results.append({
            "entity": name,
            "categories": categories,
            "examples": paper_examples
        })

# Save results
with open("entity_examples.json", "w", encoding="utf-8") as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

print("✅ Multi-example structured extraction saved to entity_examples.json")


### Extract metadata entities and relationships for one single paper

In [ ]:
# Load input files
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(ENTITY_MAP_PATH, "r", encoding="utf-8") as f:
    entity_map = json.load(f)

with open(ENTITIES_DEF_PATH, "r", encoding="utf-8") as f:
    entities_def = {e["name"]: e for e in json.load(f)["entities"]}

# Only the 8 metadata entities to process
metadata_entity_names = [
    "Academic Paper", "Author", "Journal", "Year of Publication",
    "DOI", "Keywords", "Affiliations", "Study Area"
]

def sanitize(text):
    """Safe identifier creation, converts non-str to str first."""
    txt = "" if text is None else str(text)
    return re.sub(r'\W+', '_', txt).strip('_')

# Prepare output dir
os.makedirs("Paper_KGs", exist_ok=True)

for paper_id, paper_data in metadata.items():
    kg = {"Entities": [], "Relationships": []}
    nodes = {}
    
    '''
    # First version of add_node Function
    def add_node(label, key, props=None):
        node_id = f"{label}_{sanitize(key)}"
        if node_id not in nodes:
            node = {"id": node_id, "label": label, "properties": {"value": key}}
            if props:
                node["properties"].update(props)
            nodes[node_id] = node
        return node_id
    '''
    #'''
    # Second version of add_node Function
    node_counter = 1
    node_id_map = {}

    def add_node(label, key, props=None):
        global node_counter
        unique_key = f"{label}_{sanitize(key)}"
        if unique_key not in node_id_map:
            node_id = f"meta_{paper_id}_{node_counter:03d}"
            node = {"id": node_id, "label": label, "properties": {"value": key}}
            if props:
                node["properties"].update(props)
            nodes[node_id] = node
            node_id_map[unique_key] = node_id
            node_counter += 1
        return node_id_map[unique_key]
    #'''

    def add_relationship(src, rel_type, dst):
        kg["Relationships"].append({
            "source": src, "type": rel_type, "target": dst
        })

    # 1. Academic Paper entity
    title = paper_data.get("Title", "")
    paper_node = add_node("Academic Paper", paper_id, {"title": title})

    # 2. DOI
    doi = paper_data.get("DOI", "")
    if doi:
        doi_node = add_node("DOI", doi, {"identifier": doi})
        add_relationship(paper_node, "HAS_COMPONENT", doi_node)

    # 3. Year of Publication
    year = paper_data.get("Year", "")
    year_node = add_node("Year of Publication", year, {"year": year})
    add_relationship(paper_node, "HAS_COMPONENT", year_node)

    # 4. Journal
    journal = paper_data.get("Journal", "")
    journal_node = add_node("Journal", journal)
    add_relationship(paper_node, "HAS_COMPONENT", journal_node)

    # 5. Authors
    for author in paper_data.get("Author", []):
        auth_node = add_node("Author", author)
        add_relationship(paper_node, "HAS_COMPONENT", auth_node)

    # 6. Affiliations
    aff_text = paper_data.get("Author_Affiliations", "") or ""
    # Pattern: [name1; name2] affiliation text.
    pattern = re.compile(r'\[([^\]]+)\]\s*([^\[]+?)(?=(?:\[[^\]]+\])|$)')
    for match in pattern.finditer(aff_text):
        names = match.group(1)
        aff_str = match.group(2).strip().rstrip('.')
        # Split into institution, department, address
        parts = [p.strip() for p in aff_str.split(',')]
        institution = parts[0]
        department = parts[1] if len(parts) > 1 else ""
        address = ", ".join(parts[2:]) if len(parts) > 2 else ""
        aff_node = add_node("Affiliations", institution, {
            "institution_name": institution,
            "department": department,
            "address": address
        })
        # Link paper to affiliation
        add_relationship(paper_node, "HAS_COMPONENT", aff_node)
        # Link each author to this affiliation
        for name in names.split(';'):
            name = name.strip()
            if name:
                auth_node = add_node("Author", name)
                add_relationship(auth_node, "AFFILIATED_WITH", aff_node)

    # 7. Keywords
    for kw in paper_data.get("Keywords", []):
        kw_node = add_node("Keywords", kw, {"keyword_list": kw})
        add_relationship(paper_node, "HAS_COMPONENT", kw_node)

    # 8. Study Area
    for area in paper_data.get("Study Area", []):
        area_node = add_node("Study Area", area, {"geographical_scope": area})
        add_relationship(paper_node, "HAS_COMPONENT", area_node)

    # Collect unique entities
    kg["Entities"] = list(nodes.values())

    # Write JSON for this paper
    out_file = os.path.join("Paper_KGs", f"kg_Metadata_{paper_id}.json")
    with open(out_file, "w", encoding="utf-8") as fw:
        json.dump(kg, fw, indent=2, ensure_ascii=False)

    print(f"✔️  Generated KG for paper {paper_id} at {out_file}")


### Extract entities for a single paper

In [ ]:

# Load files
#with open(EXTRACTED_TEST_PATH,    encoding="utf-8") as f: extracted    = json.load(f)
with open(EXTRACTED_PATH,         encoding="utf-8") as f: extracted    = json.load(f)
with open(ENTITIES_DEF_PATH,      encoding="utf-8") as f: entities_def = {e["name"]: e for e in json.load(f)["entities"]}
with open(ENTITY_MAP_PATH,        encoding="utf-8") as f: entity_map   = json.load(f)
with open(EXAMPLE_ENTITIES_PATH,  encoding="utf-8") as f: few_shot     = json.load(f)

# Build few‑shot per entity
few_shot_by_entity = {}
for e in few_shot:
    few_shot_by_entity[e["entity"]] = e["examples"]

# Entities to extract (skip Metadata ones)
to_extract = [n for n in entity_map if "Metadata" not in entity_map[n]]

os.makedirs("Paper_KGs", exist_ok=True)

def make_prompt(entity_name, properties, categories, input_text):
    # assemble few‑shot
    shots = few_shot_by_entity.get(entity_name, [])[:3]
    shot_block = ""
    for ex in shots:
        shot_block += "\n\n###\nInput:\n" + ex["input"] + "\n\nOutput:\n" \
                    + json.dumps(ex["expected_output"], indent=2, ensure_ascii=False)
    # property template
    props_tpl = ",\n        ".join(f'"{p}": "..."' for p in properties)
    return f"""
You are an expert in hydrology and scientific knowledge extraction for knowledge graph construction.

Your task: given the following **{entity_name}**‐related snippets, identify all instances of **{entity_name}** and pull out any of these properties if present: {properties}.

What to do:
- Identify all relevant entities of type **{entity_name}** in each example input.
- Return each entity as a short, semantically meaningful name.
- Avoid copying long sentences or including locations/datasets unless essential for the entity **{entity_name}**.
- Also extract these properties if available: {properties}
- If no relevant entities are found, respond with an empty list.
- If importance_level is part of the property list, and it is not present in the input, infer it from the context.

Return JSON with exactly this schema:
{{
  "entities": [
    {{
      "name": "...",
      "category": "{entity_name}",
      "properties": {{
        {props_tpl}
      }}
    }}
  ]
}}

Few‐shot examples:{shot_block}

### Now your turn

Input text (from categories: {categories}):
\"\"\"
{input_text}
\"\"\"
"""

for paper in extracted:
    pid = paper["Paper ID"]
    # collect each paper's KG
    out_kg = {"Paper ID": pid, "Entities": []}

    for entity_name in to_extract:
        props     = entities_def[entity_name]["possible_properties"]
        cats      = entity_map[entity_name]
        # stitch together only those categories we have text for
        snippets = [paper[c] for c in cats if paper.get(c)]
        if not snippets: 
            continue
        input_text = "\n\n".join(snippets)

        prompt = make_prompt(entity_name, props, cats, input_text)

        try:
            response = client.chat.completions.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user",  "content": prompt}
                ],
                temperature=0.0,
                response_format={"type": "json_object"}
            )
            parsed_output = response.choices[0].message.content.strip()
            result = json.loads(parsed_output)
        except Exception as e:
            result = {"error": str(e)}

        out_kg["Entities"].append({
            "entity": entity_name,
            "result": result
        })

    #'''
    # Create unique entity IDs
    entity_counter = 1
    for entity_group in out_kg["Entities"]:
        for entity in entity_group["result"]["entities"]:
            entity["id"] = f"ent_{pid}_{entity_counter:03d}"
            entity_counter += 1
    #'''

    # write per‐paper output
    with open(f"Paper_KGs/kg_Entities_{pid}.json", "w", encoding="utf-8") as fw:
        json.dump(out_kg, fw, indent=2, ensure_ascii=False)
    
    print(f"✅ Done {pid}")

### Extract relationships for a single paper

In [ ]:
# Load relationships map
with open(ENTITY_REL_MAP_PATH, encoding="utf-8") as f:
    relationship_map = json.load(f)

# Utility to get entities by category
def get_entities_by_category(category_name, kg_entities):
    for item in kg_entities:
        if item["entity"] == category_name:
            return item["result"]["entities"]
    return []

# Function to check relationships using OpenAI API
def check_relationship_via_llm(entity1, entity2, relation_type):
    def format_entity(entity):
        props = entity['properties']
        formatted_props = "\n".join(
            f"- {key}: \"{value}\""
            for key, value in props.items()
            if value and key != "importance_level"
        )
        return f"Entity ({entity['category']}): {entity['name']}\n{formatted_props}"

    entity1_formatted = format_entity(entity1)
    entity2_formatted = format_entity(entity2)

    prompt = f"""
Given these two entities:

Entity 1 {entity1_formatted}

Entity 2 {entity2_formatted}

Does Entity 1 '{relation_type}' Entity 2? Return a JSON with the format:
{{"answer": "yes"}} or {{"answer": "no"}}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "system", "content": "You are an expert in hydrology and scientific knowledge extraction for knowledge graph construction."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        answer_json = json.loads(response.choices[0].message.content.strip().lower())
        return answer_json.get('answer') == 'yes'
    except Exception as e:
        print(f"Error evaluating relationship: {e}")
        return False

# Function to get entity ID based on name and category
def get_entity_id(entity_name, entity_category, kg_entities):
    for group in kg_entities:
        for ent in group["result"]["entities"]:
            if ent["name"] == entity_name and group["entity"] == entity_category:
                return ent["id"]
    return None

# Main function to generate relationships for a given paper ID
def generate_intra_paper_relationships(paper_id):
    # Paths
    KG_ENTITIES_PATH = f"Paper_KGs/kg_Entities_{paper_id}.json"
    OUTPUT_RELATIONS_PATH = f"Paper_KGs/Kg_Relationships_{paper_id}.json"

    # Load paper entities
    with open(KG_ENTITIES_PATH, encoding="utf-8") as f:
        kg_entities = json.load(f)["Entities"]

    relationships = []

    for rel_type, pairs in relationship_map.items():
        for pair in pairs:
            source_entities = get_entities_by_category(pair["source"], kg_entities)
            target_entities = get_entities_by_category(pair["target"], kg_entities)

            '''
            # First version of: Check if the relationship exists between the source and target entities
            for src in source_entities:
                for tgt in target_entities:
                    if check_relationship_via_llm(src, tgt, rel_type):
                        relationships.append({
                            "source": f"{src['category']}_{src['name']}",
                            "type": rel_type,
                            "target": f"{tgt['category']}_{tgt['name']}"
                        })
            '''
            #'''
            # Second version of: Check if the relationship exists between the source and target entities
            for src in source_entities:
                for tgt in target_entities:
                    if check_relationship_via_llm(src, tgt, rel_type):
                        src_id = get_entity_id(src["name"], src["category"], kg_entities)
                        tgt_id = get_entity_id(tgt["name"], tgt["category"], kg_entities)
                        if src_id and tgt_id:
                            relationships.append({
                                "source": src_id,
                                "type": rel_type,
                                "target": tgt_id
                            })
            #'''

    # Save relationships
    with open(OUTPUT_RELATIONS_PATH.format(pid=paper_id), "w", encoding="utf-8") as fw:
        json.dump({"Relationships": relationships}, fw, indent=2, ensure_ascii=False)

    print(f"✅ Relationships generated and saved to {OUTPUT_RELATIONS_PATH.format(pid=paper_id)}")

# Generate relationships for each paper
with open(EXTRACTED_PATH, encoding="utf-8") as f: extracted = json.load(f)
for paper in extracted:
    paper_id = paper["Paper ID"]
    try:
        generate_intra_paper_relationships(paper_id)
    except Exception as e:
        print(f"❌ Error generating relationships for paper {paper_id}: {e}")
